In [ ]:
"""
PHISHING DETECTION - 1.5M URLs (UPDATED DATASET)
Multi-class Classification: phishing, benign, defacement, malware
Trains both Random Forest and Neural Network
Optimized for large datasets on Kaggle
"""

# ============================================
# SETUP & IMPORTS
# ============================================

!pip install -q scikit-learn

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib
import gc
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Memory optimization
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# ============================================
# 1. LOAD PRE-SPLIT DATA FROM KAGGLE INPUT
# ============================================

# NOTE: On Kaggle, datasets are auto-mounted at /kaggle/input/your-dataset-name/
# Replace with your actual Kaggle dataset name
# IMPORTANT: Make sure path ends with '/' (forward slash)!

DATASET_PATH = '/kaggle/input/phishing-dataset-1-5m/'  # ← Change to YOUR dataset name (must end with /)

print("📥 Loading pre-split dataset from Kaggle...")

try:
    # Try Kaggle path first
    train_df = pd.read_csv(DATASET_PATH + 'train.csv')
    val_df = pd.read_csv(DATASET_PATH + 'validation.csv')
    test_df = pd.read_csv(DATASET_PATH + 'test.csv')
    print("✅ Loaded from Kaggle dataset")
except:
    # Fallback for Colab (upload files)
    print("⚠️  Kaggle path not found, using file upload...")
    from google.colab import files
    uploaded = files.upload()
    
    train_df = pd.read_csv('train.csv')
    val_df = pd.read_csv('validation.csv')
    test_df = pd.read_csv('test.csv')
    print("✅ Loaded from uploaded files")

# ============================================
# CONVERT LABELS TO NUMERIC (0, 1, 2, 3)
# ============================================

print(f"\n🔄 Converting text labels to numeric...")

# Original label distribution
print(f"\nOriginal labels:")
for label in sorted(train_df['label'].unique()):
    count = (train_df['label'] == label).sum()
    print(f"   {label}: {count:,}")

# Create label mapping: benign=0, defacement=1, malware=2, phishing=3
label_map = {
    'benign': 0,
    'defacement': 1,
    'malware': 2,
    'phishing': 3
}

# Reverse mapping for display
label_names = ['benign', 'defacement', 'malware', 'phishing']

def convert_label(label):
    """Convert text label to numeric"""
    if isinstance(label, str):
        return label_map.get(label.lower(), 0)
    else:
        return int(label)

train_df['label'] = train_df['label'].apply(convert_label)
val_df['label'] = val_df['label'].apply(convert_label)
test_df['label'] = test_df['label'].apply(convert_label)

print(f"\n✅ Converted to numeric labels:")
for i, name in enumerate(label_names):
    count = (train_df['label'] == i).sum()
    print(f"   {i} ({name}): {count:,}")

print(f"\n📊 Dataset splits:")
print(f"   Train: {len(train_df):,} ({len(train_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%)")
print(f"   Val:   {len(val_df):,} ({len(val_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%)")
print(f"   Test:  {len(test_df):,} ({len(test_df)/(len(train_df)+len(val_df)+len(test_df))*100:.1f}%)")

print(f"\n📊 Label distribution per split:")
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"\n{split_name}:")
    for i, name in enumerate(label_names):
        count = (split_df['label'] == i).sum()
        pct = (count / len(split_df) * 100)
        print(f"   {name}: {count:,} ({pct:.2f}%)")

# Check URL length distribution
print(f"\n📏 URL Length Statistics:")
all_urls = pd.concat([train_df['url'], val_df['url'], test_df['url']])
print(f"   Mean: {all_urls.str.len().mean():.0f} chars")
print(f"   Median: {all_urls.str.len().median():.0f} chars")
print(f"   Min: {all_urls.str.len().min()} chars")
print(f"   Max: {all_urls.str.len().max()} chars")

# ============================================
# 2. URL PREPROCESSING (CHARACTER ENCODING)
# ============================================

print("\n🔤 Building character vocabulary...")

# Build vocab from training set only
all_chars = set()
for url in tqdm(train_df['url'], desc="Analyzing URLs"):
    all_chars.update(url)

chars = sorted(list(all_chars))
char_to_idx = {ch: i+1 for i, ch in enumerate(chars)}  # 0 reserved for padding
vocab_size = len(chars) + 1

print(f"✅ Vocabulary size: {vocab_size}")

# Encode URLs
MAX_URL_LEN = 200  # Truncate to 200 chars

def encode_url(url):
    """Convert URL to sequence of character indices"""
    return [char_to_idx.get(c, 0) for c in url[:MAX_URL_LEN]]

print(f"\n🔧 Encoding URLs (max length: {MAX_URL_LEN})...")

# Encode in batches to save memory
def encode_batch(df, batch_size=10000):
    encoded = []
    for i in tqdm(range(0, len(df), batch_size), desc="Encoding"):
        batch = df['url'].iloc[i:i+batch_size]
        encoded.extend([encode_url(url) for url in batch])
    return encoded

X_train_encoded = encode_batch(train_df)
X_val_encoded = encode_batch(val_df)
X_test_encoded = encode_batch(test_df)

# Pad sequences
print("📦 Padding sequences...")
X_train = keras.preprocessing.sequence.pad_sequences(
    X_train_encoded, maxlen=MAX_URL_LEN, padding='post'
)
X_val = keras.preprocessing.sequence.pad_sequences(
    X_val_encoded, maxlen=MAX_URL_LEN, padding='post'
)
X_test = keras.preprocessing.sequence.pad_sequences(
    X_test_encoded, maxlen=MAX_URL_LEN, padding='post'
)

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# Free memory
del X_train_encoded, X_val_encoded, X_test_encoded, train_df, val_df, test_df
gc.collect()

print(f"\n✅ Encoding complete!")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_val shape: {X_val.shape}")
print(f"   X_test shape: {X_test.shape}")

# ============================================
# 3. MODEL 1: RANDOM FOREST (with sampling)
# ============================================

print("\n" + "="*60)
print("🌲 RANDOM FOREST CLASSIFIER (Multi-class)")
print("="*60)

# For large datasets, subsample for Random Forest to avoid memory issues
SUBSAMPLE_SIZE = 200000  # Use 200K for RF

print(f"\n📊 Subsampling for Random Forest: {SUBSAMPLE_SIZE:,} URLs")

# Stratified subsample (balanced across all 4 classes)
selected_indices = []
samples_per_class = SUBSAMPLE_SIZE // 4  # 50K per class

for class_id in range(4):
    class_indices = np.where(y_train == class_id)[0]
    n_available = len(class_indices)
    n_sample = min(samples_per_class, n_available)
    
    np.random.seed(42)
    selected = np.random.choice(class_indices, size=n_sample, replace=False)
    selected_indices.extend(selected)
    
    print(f"   Class {class_id} ({label_names[class_id]}): {n_sample:,} samples")

selected_indices = np.array(selected_indices)
np.random.shuffle(selected_indices)

X_train_rf = X_train[selected_indices]
y_train_rf = y_train[selected_indices]

print(f"\n✅ RF training set: {len(X_train_rf):,} URLs")

# Train Random Forest
print("\n🚀 Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=4,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rf_model.fit(X_train_rf, y_train_rf)

# Evaluate RF
print("\n📊 Evaluating Random Forest...")

rf_val_pred = rf_model.predict(X_val)
rf_test_pred = rf_model.predict(X_test)

rf_results = {
    'validation': {
        'accuracy': accuracy_score(y_val, rf_val_pred),
        'precision': precision_score(y_val, rf_val_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_val, rf_val_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_val, rf_val_pred, average='weighted', zero_division=0)
    },
    'test': {
        'accuracy': accuracy_score(y_test, rf_test_pred),
        'precision': precision_score(y_test, rf_test_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, rf_test_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, rf_test_pred, average='weighted', zero_division=0)
    }
}

print("\n📈 Random Forest Results:")
print(f"   Validation - Acc: {rf_results['validation']['accuracy']:.4f}, F1: {rf_results['validation']['f1']:.4f}")
print(f"   Test       - Acc: {rf_results['test']['accuracy']:.4f}, F1: {rf_results['test']['f1']:.4f}")

# Per-class accuracy
print("\n📊 Per-Class Accuracy (Test Set):")
for i, name in enumerate(label_names):
    class_mask = (y_test == i)
    class_acc = (rf_test_pred[class_mask] == i).sum() / class_mask.sum() if class_mask.sum() > 0 else 0
    print(f"   {name}: {class_acc:.4f} ({class_mask.sum():,} samples)")

# Save RF model
joblib.dump(rf_model, 'random_forest_model.pkl')
print("\n💾 Saved: random_forest_model.pkl")

# Free memory
del X_train_rf, y_train_rf, selected_indices
gc.collect()

# ============================================
# 4. MODEL 2: NEURAL NETWORK (High Dropout)
# ============================================

print("\n" + "="*60)
print("🧠 NEURAL NETWORK (Character-level CNN - Multi-class)")
print("="*60)

# Build model with HIGH DROPOUT (prevents memorization)
model = keras.Sequential([
    # Embedding layer
    layers.Embedding(vocab_size, 128, input_length=MAX_URL_LEN),
    
    # Conv blocks with high dropout
    layers.Conv1D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.Dropout(0.6),
    
    layers.Conv1D(256, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.Dropout(0.6),
    
    layers.Conv1D(256, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.GlobalMaxPooling1D(),
    layers.Dropout(0.7),
    
    # Dense layers with high dropout
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.7),
    
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.6),
    
    # Output layer: 4 classes with softmax
    layers.Dense(4, activation='softmax')
])

# Compile
initial_learning_rate = 0.001
lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate,
    decay_steps=10000,
    decay_rate=0.9,
    staircase=True
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss='sparse_categorical_crossentropy',  # Multi-class
    metrics=['accuracy']
)

print("\n🔧 Model Architecture:")
model.summary()

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_model.keras',  # Modern .keras format
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# Train
print("\n🚀 Training Neural Network...")
print("   ⚠️  High dropout (0.6-0.7) = less memorization = better generalization")

BATCH_SIZE = 256

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=20,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

# Evaluate NN
print("\n📊 Evaluating Neural Network...")

nn_val_pred = model.predict(X_val, batch_size=BATCH_SIZE).argmax(axis=1)
nn_test_pred = model.predict(X_test, batch_size=BATCH_SIZE).argmax(axis=1)

nn_results = {
    'validation': {
        'accuracy': accuracy_score(y_val, nn_val_pred),
        'precision': precision_score(y_val, nn_val_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_val, nn_val_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_val, nn_val_pred, average='weighted', zero_division=0)
    },
    'test': {
        'accuracy': accuracy_score(y_test, nn_test_pred),
        'precision': precision_score(y_test, nn_test_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, nn_test_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, nn_test_pred, average='weighted', zero_division=0)
    }
}

print("\n📈 Neural Network Results:")
print(f"   Validation - Acc: {nn_results['validation']['accuracy']:.4f}, F1: {nn_results['validation']['f1']:.4f}")
print(f"   Test       - Acc: {nn_results['test']['accuracy']:.4f}, F1: {nn_results['test']['f1']:.4f}")

# Per-class accuracy
print("\n📊 Per-Class Accuracy (Test Set):")
for i, name in enumerate(label_names):
    class_mask = (y_test == i)
    class_acc = (nn_test_pred[class_mask] == i).sum() / class_mask.sum() if class_mask.sum() > 0 else 0
    print(f"   {name}: {class_acc:.4f} ({class_mask.sum():,} samples)")

# Save NN model (modern .keras format)
model.save('neural_network_model.keras')
print("\n💾 Saved: neural_network_model.keras")

# ============================================
# 5. FINAL COMPARISON & VISUALIZATION
# ============================================

print("\n" + "="*60)
print("📊 FINAL MODEL COMPARISON")
print("="*60)

comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Neural Network'],
    'Val Accuracy': [rf_results['validation']['accuracy'], nn_results['validation']['accuracy']],
    'Val F1-Score': [rf_results['validation']['f1'], nn_results['validation']['f1']],
    'Test Accuracy': [rf_results['test']['accuracy'], nn_results['test']['accuracy']],
    'Test F1-Score': [rf_results['test']['f1'], nn_results['test']['f1']]
})

print(comparison.to_string(index=False))

# Determine best model
best_model_idx = comparison['Test F1-Score'].argmax()
best_model = comparison.iloc[best_model_idx]['Model']

print(f"\n🏆 BEST MODEL: {best_model}")
print(f"   Test Accuracy: {comparison.iloc[best_model_idx]['Test Accuracy']:.4f}")
print(f"   Test F1-Score: {comparison.iloc[best_model_idx]['Test F1-Score']:.4f}")

if comparison.iloc[best_model_idx]['Test Accuracy'] >= 0.92:
    print("   ✅ SUCCESS! Meets ≥92% accuracy requirement")
else:
    print(f"   ⚠️  Accuracy {comparison.iloc[best_model_idx]['Test Accuracy']:.1%} below 92% target")

# Save comparison
comparison.to_csv('model_comparison.csv', index=False)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
metrics = ['Val Acc', 'Test Acc', 'Val F1', 'Test F1']
rf_values = [rf_results['validation']['accuracy'], rf_results['test']['accuracy'],
             rf_results['validation']['f1'], rf_results['test']['f1']]
nn_values = [nn_results['validation']['accuracy'], nn_results['test']['accuracy'],
             nn_results['validation']['f1'], nn_results['test']['f1']]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, rf_values, width, label='Random Forest', color='steelblue')
axes[0].bar(x + width/2, nn_values, width, label='Neural Network', color='darkorange')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].axhline(y=0.92, color='r', linestyle='--', label='Target (92%)', alpha=0.7)
axes[0].legend()
axes[0].set_ylim([0.8, 1.0])
axes[0].grid(axis='y', alpha=0.3)

# Training history (NN only)
axes[1].plot(history.history['accuracy'], label='Train Acc', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Neural Network Training History')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
print("\n📊 Saved visualization: model_comparison.png")
plt.show()

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# RF confusion matrix
cm_rf = confusion_matrix(y_test, rf_test_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=True,
            xticklabels=label_names, yticklabels=label_names)
axes[0].set_title('Random Forest - Test Set')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# NN confusion matrix
cm_nn = confusion_matrix(y_test, nn_test_pred)
sns.heatmap(cm_nn, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar=True,
            xticklabels=label_names, yticklabels=label_names)
axes[1].set_title('Neural Network - Test Set')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
print("📊 Saved: confusion_matrices.png")
plt.show()

# Detailed classification reports
print("\n" + "="*60)
print("📋 DETAILED CLASSIFICATION REPORTS")
print("="*60)

print("\n🌲 Random Forest - Test Set:")
print(classification_report(y_test, rf_test_pred, target_names=label_names, digits=4))

print("\n🧠 Neural Network - Test Set:")
print(classification_report(y_test, nn_test_pred, target_names=label_names, digits=4))

# ============================================
# 6. SAVE/DOWNLOAD RESULTS
# ============================================

print("\n💾 Saving all results...")

# Try to download if on Colab
try:
    from google.colab import files
    print("📥 Downloading files from Colab...")
    files.download('random_forest_model.pkl')
    files.download('neural_network_model.keras')
    files.download('model_comparison.csv')
    files.download('model_comparison.png')
    files.download('confusion_matrices.png')
except:
    print("✅ Files saved (not on Colab, no auto-download)")

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
total_urls = X_train.shape[0] + X_val.shape[0] + X_test.shape[0]
print(f"📊 Dataset size: {total_urls:,} URLs")
print(f"🏆 Best model: {best_model}")
print(f"📈 Overall Test Accuracy: {comparison.iloc[best_model_idx]['Test Accuracy']:.4f}")
print(f"📈 Overall Test F1-Score: {comparison.iloc[best_model_idx]['Test F1-Score']:.4f}")
print(f"\n📊 Per-Class Performance (Best Model = {best_model}):")
if best_model == 'Random Forest':
    for i, name in enumerate(label_names):
        class_mask = (y_test == i)
        class_acc = (rf_test_pred[class_mask] == i).sum() / class_mask.sum() if class_mask.sum() > 0 else 0
        print(f"   {name}: {class_acc:.4f}")
else:
    for i, name in enumerate(label_names):
        class_mask = (y_test == i)
        class_acc = (nn_test_pred[class_mask] == i).sum() / class_mask.sum() if class_mask.sum() > 0 else 0
        print(f"   {name}: {class_acc:.4f}")
print("="*60)